In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from bedrock.autograd.engine import Value
from bedrock.autograd.nn import Neuron

# Logistic Regression with Micrograd

Logistic regression is an important technique for binary classification.


By combining a neuron (w*x+b) with sigmoid, we can clamp outputs between 0 and 1.

Using cross-entropy loss, we can update out neuron through backpropagation to have its weights learn when the output value should move towards 0 or 1.

## Using Micrograd

Since we have already built a neuron in our version of Karpathy's micrograd, we can use these building blocks to build a logistic regression "model".

To start, we need to update the `Value` class with a sigmoid method and a log method. 

### Sigmoid function

The sigmoid function bounds its input between 0 and 1. This is very similar to tanh

### Log function

We need the log function to calculate the cross-entropy loss

## Testing with basic scalars

First we can test that the sigmoid and loss functions work using our `Value` objects. Afterwards we can move to using `Neuron`s and performing gradient descent.

In [ ]:
# Initialize "Neuron", input value and target
w, b = Value(2.0), Value(-4.0)
x, y = 3.0, 1.0

In [ ]:
pred = (w * x + b).sigmoid()
# Cross entropy loss : -1/N * (y * log(y_hat) + (1-y) * log(1-y_hat))
loss = -(y * pred.log() + (1 - y) * (1 - pred).log())

In [ ]:
loss.backward()
# roughly equal to [0.88, -0.36, -0.12]
print(pred.data, w.grad, b.grad)

Now with real data

In [ ]:
data = [(0.5, 0), (1.0, 0), (1.5, 0), (2.5, 1), (3.0, 1), (3.5, 1)]
# w, b = Value(0.0), Value(0.0)
lr = 0.1

n = Neuron(1, act="sigmoid")


def cross_entropy(pred: Value, target: int):
    return -(target * pred.log() + (1 - target) * (1 - pred).log())


for step in range(100):
    # 1. average the loss over all 6 examples
    avg_loss: Value = sum((cross_entropy(n([x]), y) for x, y in data)) / len(data)
    # 2. zero w.grad and b.grad
    n.zero_grad()

    # 3. backward
    avg_loss.backward()

    # 4. update w and b
    for param in n.parameters():
        param.data -= lr * param.grad

    if step % 10 == 0:
        print(step, avg_loss.data, n.w[0].data, n.b.data)

## Running with numpy-backed LogisticRegression model
The scipy style `fit` method splits features and targets.

The decision boundary for single feature derived from the following:
- Decision flips from *0 <-> 1* at `sigmoid(score)=0.5`, which is when `score=0`
- `score = weight * x + bias = 0`
- `x = -bias / weight`

In [ ]:
import numpy as np
from bedrock.models.logistic_regression import LogisticRegression

X = np.array([[0.5], [1.0], [1.5], [2.5], [3.0], [3.5]])
y = np.array([0, 0, 0, 1, 1, 1])

model = LogisticRegression(lr=0.5, epochs=1000).fit(X, y)
print("weights:", model.w, "bias:", model.b)
print("boundary:", -model.b / model.w[0])
print("probabilities:", model.predict_proba(X).round(3))